# Bottleneck & Roofline Analysis

In [ ]:
import sys; sys.path.insert(0, '../..')
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collection.track1_ggml.profiler_wrapper import parse_profiler_output
from analysis.bottleneck import BottleneckAnalyzer
from analysis.device_specs import DEVICE_SPECS
from collections import Counter

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300})

In [ ]:
records = parse_profiler_output(
    '../../data/raw/rpi4/rpi4_qwen2.5_1.5b_profile.jsonl',
    'rpi4_analysis', 'rpi4', 'qwen2.5-1.5b', 'qwen', 'q4_k_m', 7)

analyzer = BottleneckAnalyzer()

# Classify
classifications = [analyzer.classify(r) for r in records if r.latency_us > 10]
counts = Counter(classifications)
print('RPi4 Bottleneck Classification:')
for k, v in counts.most_common():
    print(f'  {k}: {v} ({v/len(classifications)*100:.1f}%)')

## Roofline Plot

In [ ]:
roofline_df = analyzer.compute_roofline(records)
spec = DEVICE_SPECS['rpi4']

fig, ax = plt.subplots(figsize=(10,7))
oi_range = np.logspace(-2, 4, 200)
ceiling = np.minimum(spec.peak_gflops, oi_range * spec.peak_bandwidth_gb_s)
ax.loglog(oi_range, ceiling, 'k-', linewidth=2, label='Roofline ceiling')

for _, row in roofline_df.iterrows():
    if row['achieved_gflops'] > 0 and row['operational_intensity'] > 0:
        ax.scatter(row['operational_intensity'], row['achieved_gflops'], s=30, alpha=0.5)

ax.set_xlabel('Operational Intensity (FLOP/Byte)')
ax.set_ylabel('Achieved GFLOPS')
ax.set_title(f'RPi4 Roofline — Qwen2.5-1.5B (peak={spec.peak_gflops} GFLOPS, BW={spec.peak_bandwidth_gb_s} GB/s)')
ax.legend()
plt.tight_layout()
plt.savefig('../../claudedocs/figures/nb4_roofline.pdf', bbox_inches='tight')
plt.show()